# Crystals: silicon in a Gaussian basis and in plane waves

Bulk silicon with GTH pseudopotentials, solved by the Gaussian engine on a k-mesh and by the plane-wave engine, then forces on a displaced atom and the lattice constant from the stress tensor. The GTH tables are read from an installed PySCF (`pip install dftax[periodic]`) or from `$DFTAX_GTH_DATA`.

In [ ]:
import jax; jax.config.update("jax_enable_x64", True)
import numpy as np
from dftax import forces, scf
from dftax.energy.xc import LDA
from dftax.pbc import Cell, kmesh, kpoint_ks, planewaves, scf_pw, stress, uniform

HARTREE_EV, AU_GPA = 27.211386, 29421.0158


def silicon(alat, shift=0.0):
    """The two-atom primitive cell of diamond silicon (Angstrom); `shift`
    moves the second atom along x."""
    a = 0.5 * alat * np.array([[0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 0.0]])
    q = alat / 4
    return Cell.from_xyz(f"Si 0 0 0; Si {q + shift} {q} {q}", a,
                         basis="gth-dzvp", pseudo="gth-pade")


def gap(res):
    """The band gap on the k-mesh (eV)."""
    eps, occ = np.asarray(res.eps), np.asarray(res.occ)
    return (eps[occ < 1.0].min() - eps[occ > 1.0].max()) * HARTREE_EV


cell = silicon(5.431)
kpts = kmesh(4, 4, 4)
print(f"{cell.nelectron} valence electrons, {len(kpts[0])} k-points "
      "(4x4x4, time-reversal pairs folded)")

## Gaussian basis on a k-mesh

`kpoint_ks` builds the k-point functional on a uniform real-space grid (the Hartree term and the XC quadrature live on it); `scf` solves it. The same calculation at the Gamma point alone is `KS(cell, xc, grid=grid)`, the engine for large supercells.

In [ ]:
grid = uniform(mesh=(21, 21, 21))
res_g = scf(kpoint_ks(cell, LDA(), grid=grid, kpts=kpts))
print(f"Gaussian (gth-dzvp): E = {res_g.e_tot:.6f} Ha, gap {gap(res_g):.3f} eV, "
      f"converged {res_g.converged}")

## Plane waves

The same potential and k-mesh in plane waves up to 20 Ha. The plane-wave energy is the lower one: with the same Hamiltonian, the difference is what the Gaussian basis is missing.

In [ ]:
pw = planewaves(20.0)
res_pw = scf_pw(cell, LDA(), pw, kpts=kpts)
print(f"plane waves (20 Ha): E = {res_pw.e_tot:.6f} Ha, gap {gap(res_pw):.3f} eV, "
      f"converged {res_pw.converged}")
print(f"E(Gaussian) - E(plane waves) = {1000 * (res_g.e_tot - res_pw.e_tot):.2f} mHa")

## Forces

Displace the second atom by 0.05 Angstrom along x. `forces` takes the result of either engine (the Gaussian one also needs its grid); the two forces sum to zero.

In [ ]:
moved = silicon(5.431, shift=0.05)
F_g = np.asarray(forces(moved, LDA(), scf(kpoint_ks(moved, LDA(), grid=grid, kpts=kpts)),
                        grid=grid))
F_pw = np.asarray(forces(moved, LDA(), scf_pw(moved, LDA(), pw, kpts=kpts)))
print("force on the moved atom (Ha/Bohr)")
print("  Gaussian:   ", np.round(F_g[1], 5), "  net", f"{np.abs(F_g.sum(0)).max():.1e}")
print("  plane waves:", np.round(F_pw[1], 5), "  net", f"{np.abs(F_pw.sum(0)).max():.1e}")

## The lattice constant from the stress

`stress` is the derivative of the energy with respect to strain, so the pressure `p = -tr(sigma)/3` at a few lattice constants locates the equilibrium without fitting an equation of state. In plane waves it is the stress of a fixed set of plane waves; at a converged cutoff that is the stress of the converged energy.

In [ ]:
alats = np.linspace(5.30, 5.50, 5)
p = []
for alat in alats:
    c = silicon(alat)
    r = scf_pw(c, LDA(), pw, kpts=kpts)
    p.append(-np.trace(np.asarray(stress(c, LDA(), r))) / 3 * AU_GPA)
    print(f"a = {alat:.3f} A   E = {r.e_tot:.6f} Ha   p = {p[-1]:+6.2f} GPa")
a0 = np.interp(0.0, p[::-1], alats[::-1])
print(f"zero pressure at a = {a0:.3f} A (experiment 5.431 A; LDA's is typically a little smaller)")